# Dynamic Parallel IDK Scheduler



In [122]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"
RUNS_DIR = PROJECT_ROOT / "runs"
RUNS_DIR.mkdir(exist_ok=True)

import heapq
import time
from collections import Counter, deque

import numpy as np
from sklearn.ensemble import RandomForestClassifier

RESNET18 = "resnet18"
RESNET34 = "resnet34"
RESNET50 = "resnet50"
RESNET152 = "resnet152"
MODELS = (RESNET18, RESNET34, RESNET50, RESNET152)
HEAVY_MODELS = (RESNET50, RESNET152)

DYNAMIC_CONFIDENCE_THRESHOLD = 0.9
SEQUENTIAL_CONFIDENCE_THRESHOLD = 0.9

HEAVY_ROUTE_LABEL_BY_MODEL = {RESNET50: 0, RESNET152: 1}
HEAVY_MODEL_BY_ROUTE_LABEL = {
    label: model for model, label in HEAVY_ROUTE_LABEL_BY_MODEL.items()
}
SKIP_RESNET34_LABEL = 0
USE_RESNET34_LABEL = 1

## Cache Handling and Features

In [123]:
REQUIRED_CACHE_FIELDS = ("probabilities", "labels", "predictions", "times_ms", "keys")

##### Returns if the cache for the model(s) is valid

In [124]:
def validate_model_caches(caches):
    missing_models = [model for model in MODELS if model not in caches]
    if missing_models:
        raise ValueError(f"Missing model caches: {missing_models}")

    reference_cache = caches[RESNET18]
    sample_count = len(reference_cache["labels"])


    # for model, cache in caches.items():
    #     missing_fields = [field for field in REQUIRED_CACHE_FIELDS if field not in cache]
    #     if missing_fields:
    #         raise ValueError(f"{model} cache is missing: {missing_fields}")
    #     if any(len(cache[field]) != sample_count for field in REQUIRED_CACHE_FIELDS):
    #         raise ValueError(f"{model} cache arrays have different lengths")
    #     if not np.array_equal(cache["labels"], reference_cache["labels"]):
    #         raise ValueError(f"{model} labels do not match {RESNET18}")
    #     if not np.array_equal(cache["keys"], reference_cache["keys"]):
    #         raise ValueError(f"{model} keys do not match {RESNET18}")

    return sample_count

##### Loads the model cache from file path

In [125]:
def load_model_caches(paths_by_model):
    caches = {}
    for model, path in paths_by_model.items():
        with np.load(path) as data:
            caches[model] = {name: data[name] for name in data.files}
    validate_model_caches(caches)
    return caches

##### Returns 'n' sample cache instead of entire dataset

In [126]:
def take_first_samples(caches, sample_count):
    # if not isinstance(sample_count, (int, np.integer)) or sample_count <= 0:
        # raise ValueError("sample_count must be a positive integer")
    sample_count = min(int(sample_count), validate_model_caches(caches))
    sliced_caches = {
        model: {
            field: value[:sample_count] if field in REQUIRED_CACHE_FIELDS else value
            for field, value in cache.items()
        }
        for model, cache in caches.items()
    }
    validate_model_caches(sliced_caches)
    return sliced_caches

##### Returns highest probability of a sample 

In [127]:
def max_confidence(cache):
    return np.asarray(cache["probabilities"]).max(axis=1)

##### Returns combined feature table for confidence, entropy, margin

In [128]:
def extract_probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    # if probabilities.ndim != 2 or probabilities.shape[1] < 2:
    #     raise ValueError("probabilities must have shape [sample_count, class_count >= 2]")
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)

##### Returns # of classification that a model did

In [129]:
def model_counts(model_names, models=MODELS):
    return {model: int(np.count_nonzero(model_names == model)) for model in models}

##### Returns # of execution each model started

In [130]:
def counter_counts(counter, models):
    return {model: int(counter[model]) for model in models}

##### Returns latency info

In [131]:
def latency_metrics(latencies_ms):
    return {
        "mean_latency_ms": float(latencies_ms.mean()), "median_latency_ms": float(np.median(latencies_ms)), "p95_latency_ms": float(np.percentile(latencies_ms, 95))
    }

## Heavy-Model Router
#### Trains and evaluate the Random Forest that chooses between ResNet-50 and ResNet-152 when both early models are IDK.

##### Return's # of 0s (ResNet-50) and 1s (ResNet-152)

In [132]:
def heavy_route_counts(route_labels):
    return {
        model: int(np.count_nonzero(route_labels == HEAVY_ROUTE_LABEL_BY_MODEL[model]))
        for model in HEAVY_MODELS
    }

##### Creates training data for Random Forest to train on. Returns 'router_feature' (Sample features of ResNet-18 & ResNet-34), 'route_labels' (Target label 0 or 1 for router), 'included_uncertain_sample_count' (# of samples where ResNet-18 & ResNet-34 failed), 'excluded_sample_count (# of samples not eligible for heavy models)

In [133]:
def build_heavy_router_dataset(caches, confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD, require_correct=False):
    validate_model_caches(caches)
    resnet18_features = extract_probability_features(caches[RESNET18]["probabilities"])
    resnet34_features = extract_probability_features(caches[RESNET34]["probabilities"])
    true_labels = caches[RESNET18]["labels"]

    both_early_models_are_uncertain = (
        (resnet18_features[:, 0] < confidence_threshold)
        & (resnet34_features[:, 0] < confidence_threshold)
    )
    resnet50_is_confident = max_confidence(caches[RESNET50]) >= confidence_threshold
    resnet152_is_confident = max_confidence(caches[RESNET152]) >= confidence_threshold

    if require_correct:
        resnet50_is_eligible = resnet50_is_confident & (
            caches[RESNET50]["predictions"] == true_labels
        )
        resnet152_is_eligible = resnet152_is_confident & (
            caches[RESNET152]["predictions"] == true_labels
        )
    else:
        resnet50_is_eligible = resnet50_is_confident
        resnet152_is_eligible = resnet152_is_confident

    included = both_early_models_are_uncertain & (
        resnet50_is_eligible | resnet152_is_eligible
    )
    router_features = np.column_stack(
        [resnet18_features[included], resnet34_features[included]]
    )
    route_labels = np.where(
        resnet50_is_eligible[included],
        HEAVY_ROUTE_LABEL_BY_MODEL[RESNET50],
        HEAVY_ROUTE_LABEL_BY_MODEL[RESNET152],
    ).astype(np.int64)

    included_uncertain_sample_count = int(both_early_models_are_uncertain.sum())
    excluded_sample_count = included_uncertain_sample_count - len(route_labels)
    return router_features, route_labels, included_uncertain_sample_count, excluded_sample_count

In [ ]:
def train_heavy_router(training_cache_sets, confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD, require_correct=False):
    feature_parts = []
    route_label_parts = []
    included_uncertain_sample_count = 0
    excluded_sample_count = 0

    for caches in training_cache_sets:
        features, labels, uncertain_count, excluded_count = build_heavy_router_dataset(
            caches, confidence_threshold, require_correct
        )
        included_uncertain_sample_count += uncertain_count
        excluded_sample_count += excluded_count
        if len(labels):
            feature_parts.append(features)
            route_label_parts.append(labels)

    if not route_label_parts:
        raise ValueError("No eligible samples were found for heavy-router training")

    training_features = np.concatenate(feature_parts)
    training_route_labels = np.concatenate(route_label_parts)
    route_counts = heavy_route_counts(training_route_labels)
    requirement = (
        "meets the threshold and predicts correctly"
        if require_correct
        else "meets the confidence threshold"
    )

    print("Heavy-router training samples:", len(training_route_labels))
    print("ResNet-50 route labels:", route_counts[RESNET50])
    print("ResNet-152 route labels:", route_counts[RESNET152])
    print(f"Excluded samples (neither heavy model {requirement}):", excluded_sample_count)
    print("Samples where both early models are below the threshold:", included_uncertain_sample_count)

    heavy_router = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
    )
    heavy_router.fit(training_features, training_route_labels)
    return heavy_router

In [135]:



def evaluate_heavy_router(
    caches,
    heavy_router,
    confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD,
    require_correct=False,
):
    features, target_labels, uncertain_count, excluded_count = (
        build_heavy_router_dataset(caches, confidence_threshold, require_correct)
    )
    predicted_labels = (
        np.asarray(heavy_router.predict(features), dtype=np.int64)
        if len(target_labels)
        else np.empty(0, dtype=np.int64)
    )
    route_label_accuracy = (
        float((predicted_labels == target_labels).mean())
        if len(target_labels)
        else float("nan")
    )
    confusion_matrix = np.zeros((2, 2), dtype=np.int64)
    if len(target_labels):
        np.add.at(confusion_matrix, (target_labels, predicted_labels), 1)

    result = {
        "both_early_uncertain_sample_count": uncertain_count,
        "evaluated_sample_count": len(target_labels),
        "excluded_sample_count": excluded_count,
        "route_label_accuracy": route_label_accuracy,
        "target_route_count_by_model": heavy_route_counts(target_labels),
        "predicted_route_count_by_model": heavy_route_counts(predicted_labels),
        "confusion_matrix": confusion_matrix,
    }

    print("Samples where both early models are below the threshold:", uncertain_count)
    print("Samples with a heavy-route target:", len(target_labels))
    print("Excluded samples without an eligible heavy model:", excluded_count)
    print("Heavy-route label accuracy:", route_label_accuracy)
    print("Target route counts:", result["target_route_count_by_model"])
    print("Predicted route counts:", result["predicted_route_count_by_model"])
    print("Confusion matrix (rows=target, columns=predicted; ResNet-50 then ResNet-152):")
    print(confusion_matrix)
    return result

In [136]:
def train_sequential_skip_router(
    training_cache_sets,
    confidence_threshold=SEQUENTIAL_CONFIDENCE_THRESHOLD,
):
    feature_parts = []
    skip_label_parts = []

    for caches in training_cache_sets:
        validate_model_caches(caches)
        resnet18_is_uncertain = max_confidence(caches[RESNET18]) < confidence_threshold
        feature_parts.append(
            extract_probability_features(
                caches[RESNET18]["probabilities"][resnet18_is_uncertain]
            )
        )
        skip_label_parts.append(
            np.where(
                max_confidence(caches[RESNET34])[resnet18_is_uncertain]
                < confidence_threshold,
                SKIP_RESNET34_LABEL,
                USE_RESNET34_LABEL,
            )
        )

    skip_router = RandomForestClassifier(
        n_estimators=50,
        max_depth=4,
        min_samples_leaf=40,
        class_weight="balanced",
        random_state=42,
        n_jobs=1,
    )
    skip_router.fit(np.concatenate(feature_parts), np.concatenate(skip_label_parts))
    return skip_router


def run_sequential_baseline(caches, skip_router):
    sample_count = validate_model_caches(caches)
    true_labels = caches[RESNET18]["labels"]
    resnet18_confidence = max_confidence(caches[RESNET18])
    resnet34_confidence = max_confidence(caches[RESNET34])

    final_predictions = np.empty(sample_count, dtype=np.int64)
    final_prediction_models = np.empty(sample_count, dtype=object)
    latencies_ms = caches[RESNET18]["times_ms"].astype(float).copy()

    resnet18_supplies_prediction = (
        resnet18_confidence >= SEQUENTIAL_CONFIDENCE_THRESHOLD
    )
    routed_indices = np.flatnonzero(~resnet18_supplies_prediction)
    final_predictions[resnet18_supplies_prediction] = caches[RESNET18]["predictions"][
        resnet18_supplies_prediction
    ]
    final_prediction_models[resnet18_supplies_prediction] = RESNET18

    resnet34_is_skipped = np.zeros(sample_count, dtype=bool)
    if len(routed_indices):
        routing_start = time.perf_counter()
        skip_labels = skip_router.predict(
            extract_probability_features(caches[RESNET18]["probabilities"][routed_indices])
        )
        routing_time_ms = (time.perf_counter() - routing_start) * 1000.0
        latencies_ms[routed_indices] += routing_time_ms / len(routed_indices)
        resnet34_is_skipped[routed_indices] = skip_labels == SKIP_RESNET34_LABEL

    resnet34_is_executed = (~resnet18_supplies_prediction) & ~resnet34_is_skipped
    resnet34_supplies_prediction = (
        resnet34_is_executed
        & (resnet34_confidence >= SEQUENTIAL_CONFIDENCE_THRESHOLD)
    )
    resnet152_is_executed = resnet34_is_skipped | (
        resnet34_is_executed & ~resnet34_supplies_prediction
    )

    latencies_ms[resnet34_is_executed] += caches[RESNET34]["times_ms"][
        resnet34_is_executed
    ]
    latencies_ms[resnet152_is_executed] += caches[RESNET152]["times_ms"][
        resnet152_is_executed
    ]
    final_predictions[resnet34_supplies_prediction] = caches[RESNET34]["predictions"][
        resnet34_supplies_prediction
    ]
    final_predictions[resnet152_is_executed] = caches[RESNET152]["predictions"][
        resnet152_is_executed
    ]
    final_prediction_models[resnet34_supplies_prediction] = RESNET34
    final_prediction_models[resnet152_is_executed] = RESNET152

    total_serial_time_ms = float(latencies_ms.sum())
    return {
        "sample_count": sample_count,
        "accuracy": float((final_predictions == true_labels).mean()),
        "total_serial_time_ms": total_serial_time_ms,
        "throughput_fps": sample_count / (total_serial_time_ms / 1000.0),
        **latency_metrics(latencies_ms),
        "final_prediction_count_by_model": model_counts(final_prediction_models),
    }

## Parallel Worker Reuse Simulation

In [137]:
def simulate_parallel_scheduler(
    caches,
    heavy_router,
    confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD,
    max_in_flight=None,
):
    sample_count = validate_model_caches(caches)
    if max_in_flight is not None:
        if not isinstance(max_in_flight, (int, np.integer)) or max_in_flight <= 0:
            raise ValueError("max_in_flight must be None or a positive integer")
        max_in_flight = int(max_in_flight)

    true_labels = caches[RESNET18]["labels"]
    resnet18_features = extract_probability_features(caches[RESNET18]["probabilities"])
    resnet34_features = extract_probability_features(caches[RESNET34]["probabilities"])
    early_confidence = {
        RESNET18: resnet18_features[:, 0],
        RESNET34: resnet34_features[:, 0],
    }
    predicted_heavy_routes = np.asarray(
        heavy_router.predict(np.column_stack([resnet18_features, resnet34_features])),
        dtype=np.int64,
    )

    start_times_ms = np.full(sample_count, np.nan)
    end_times_ms = np.full(sample_count, np.nan)
    final_predictions = np.full(sample_count, -1, dtype=np.int64)
    final_prediction_models = np.full(sample_count, None, dtype=object)
    early_model_completed = {
        model: np.zeros(sample_count, dtype=bool) for model in (RESNET18, RESNET34)
    }
    early_model_is_uncertain = {
        model: np.zeros(sample_count, dtype=bool) for model in (RESNET18, RESNET34)
    }
    heavy_job_is_queued = np.zeros(sample_count, dtype=bool)

    active_sample_by_model = {model: None for model in MODELS}
    queued_samples_by_model = {
        RESNET34: deque(),
        RESNET50: deque(),
        RESNET152: deque(),
    }
    completion_events = []
    event_sequence = 0
    next_sample_index = 0
    in_flight_count = 0
    completed_sample_count = 0
    simulation_time_ms = 0.0

    execution_count_by_model = Counter()
    busy_time_ms_by_model = Counter()
    canceled_queue_count_by_model = Counter()
    unused_completion_count_by_model = Counter()
    heavy_route_count_by_model = Counter()

    def start_job(model, sample_index, current_time_ms):
        nonlocal event_sequence
        duration_ms = float(caches[model]["times_ms"][sample_index])
        active_sample_by_model[model] = sample_index
        execution_count_by_model[model] += 1
        busy_time_ms_by_model[model] += duration_ms
        event_sequence += 1
        heapq.heappush(
            completion_events,
            (current_time_ms + duration_ms, event_sequence, model, sample_index),
        )

    def dispatch(model, current_time_ms):
        if active_sample_by_model[model] is not None:
            return
        while queued_samples_by_model[model]:
            sample_index = queued_samples_by_model[model].popleft()
            if final_prediction_models[sample_index] is not None:
                canceled_queue_count_by_model[model] += 1
                continue
            start_job(model, sample_index, current_time_ms)
            return

    def admit_sample(current_time_ms):
        nonlocal next_sample_index, in_flight_count
        if active_sample_by_model[RESNET18] is not None or next_sample_index >= sample_count:
            return
        if max_in_flight is not None and in_flight_count >= max_in_flight:
            return
        sample_index = next_sample_index
        next_sample_index += 1
        in_flight_count += 1
        start_times_ms[sample_index] = current_time_ms
        start_job(RESNET18, sample_index, current_time_ms)
        queued_samples_by_model[RESNET34].append(sample_index)

    def finalize(sample_index, model, current_time_ms):
        nonlocal in_flight_count, completed_sample_count
        if final_prediction_models[sample_index] is not None:
            return
        final_prediction_models[sample_index] = model
        final_predictions[sample_index] = caches[model]["predictions"][sample_index]
        end_times_ms[sample_index] = current_time_ms
        in_flight_count -= 1
        completed_sample_count += 1

    admit_sample(0.0)
    dispatch(RESNET34, 0.0)

    while completed_sample_count < sample_count or completion_events:
        if not completion_events:
            raise RuntimeError("Scheduler stopped with unfinished samples")

        simulation_time_ms = completion_events[0][0]
        simultaneous_completions = []
        while completion_events and completion_events[0][0] == simulation_time_ms:
            simultaneous_completions.append(heapq.heappop(completion_events))

        completed_models_by_sample = {}
        for _, _, model, sample_index in simultaneous_completions:
            if active_sample_by_model[model] != sample_index:
                raise RuntimeError("Unexpected worker completion")
            active_sample_by_model[model] = None

            if final_prediction_models[sample_index] is not None:
                unused_completion_count_by_model[model] += 1
                continue

            completed_models_by_sample.setdefault(sample_index, []).append(model)
            if model in early_confidence:
                early_model_completed[model][sample_index] = True
                early_model_is_uncertain[model][sample_index] = (
                    early_confidence[model][sample_index] < confidence_threshold
                )

        # Resolve all equal-time completions before dispatching new work.
        for sample_index, completed_models in completed_models_by_sample.items():
            completed_models = set(completed_models)
            if (
                RESNET18 in completed_models
                and not early_model_is_uncertain[RESNET18][sample_index]
            ):
                finalize(sample_index, RESNET18, simulation_time_ms)
                continue
            if (
                RESNET34 in completed_models
                and not early_model_is_uncertain[RESNET34][sample_index]
            ):
                finalize(sample_index, RESNET34, simulation_time_ms)
                continue
            if RESNET50 in completed_models:
                finalize(sample_index, RESNET50, simulation_time_ms)
                continue
            if RESNET152 in completed_models:
                finalize(sample_index, RESNET152, simulation_time_ms)
                continue

            both_early_models_are_uncertain = all(
                early_model_completed[model][sample_index]
                and early_model_is_uncertain[model][sample_index]
                for model in (RESNET18, RESNET34)
            )
            if both_early_models_are_uncertain and not heavy_job_is_queued[sample_index]:
                route_label = int(predicted_heavy_routes[sample_index])
                heavy_model = HEAVY_MODEL_BY_ROUTE_LABEL.get(route_label)
                if heavy_model is None:
                    raise ValueError("Heavy router must predict route label 0 or 1")
                heavy_job_is_queued[sample_index] = True
                heavy_route_count_by_model[heavy_model] += 1
                queued_samples_by_model[heavy_model].append(sample_index)

        admit_sample(simulation_time_ms)
        for model in (RESNET34, RESNET50, RESNET152):
            dispatch(model, simulation_time_ms)

    latencies_ms = end_times_ms - start_times_ms
    return {
        "sample_count": sample_count,
        "accuracy": float((final_predictions == true_labels).mean()),
        "simulation_makespan_ms": float(simulation_time_ms),
        "throughput_fps": sample_count / (simulation_time_ms / 1000.0),
        **latency_metrics(latencies_ms),
        "final_prediction_count_by_model": model_counts(final_prediction_models),
        "execution_count_by_model": counter_counts(execution_count_by_model, MODELS),
        "utilization_by_model": {
            model: float(busy_time_ms_by_model[model] / simulation_time_ms)
            for model in MODELS
        },
        "heavy_route_count_by_model": counter_counts(
            heavy_route_count_by_model, HEAVY_MODELS
        ),
        "canceled_queue_count_by_model": counter_counts(
            canceled_queue_count_by_model, MODELS
        ),
        "unused_completion_count_by_model": counter_counts(
            unused_completion_count_by_model, MODELS
        ),
    }


def cache_paths(artifact_prefix):
    return {
        model: ARTIFACTS_DIR / f"{artifact_prefix}_{model}.npz"
        for model in MODELS
    }


training_cache_sets = [
    load_model_caches(cache_paths(prefix)) for prefix in ("matched", "top")
]
test_caches = load_model_caches(cache_paths("threshold07"))
max_in_flight_options = (None, 2, 3, 4, 8, 16, 32)

heavy_router = train_heavy_router(training_cache_sets)
heavy_router_evaluation = evaluate_heavy_router(test_caches, heavy_router)
sequential_skip_router = train_sequential_skip_router(training_cache_sets)
sequential_result = run_sequential_baseline(test_caches, sequential_skip_router)

parallel_results_by_max_in_flight = {}
for max_in_flight in max_in_flight_options:
    limit_label = "unlimited" if max_in_flight is None else max_in_flight
    print(f"Running parallel scheduler (max in-flight samples: {limit_label})")
    parallel_results_by_max_in_flight[max_in_flight] = simulate_parallel_scheduler(
        test_caches, heavy_router, max_in_flight=max_in_flight
    )




def max_in_flight_label(limit):
    return "unlimited" if limit is None else str(limit)


comparison_runs = [("Sequential RF baseline", sequential_result)]
comparison_runs += [
    (
        f"Parallel scheduler (max in-flight: {max_in_flight_label(limit)})",
        parallel_results_by_max_in_flight[limit],
    )
    for limit in max_in_flight_options
]

print(
    f"{'System':<49} {'Accuracy':>8} {'Throughput (FPS)':>16} "
    f"{'Mean latency (ms)':>18} {'Median latency (ms)':>20} {'P95 latency (ms)':>17}"
)
for system_name, result in comparison_runs:
    print(
        f"{system_name:<49} {result['accuracy']:>8.3f} "
        f"{result['throughput_fps']:>16.2f} {result['mean_latency_ms']:>18.2f} "
        f"{result['median_latency_ms']:>20.2f} {result['p95_latency_ms']:>17.2f}"
    )

scheduler_detail_labels = {
    "final_prediction_count_by_model": "Final predictions by model",
    "execution_count_by_model": "Executions started by model",
    "utilization_by_model": "Worker utilization by model",
    "heavy_route_count_by_model": "Heavy routes assigned by model",
    "canceled_queue_count_by_model": "Queued jobs canceled before execution",
    "unused_completion_count_by_model": (
        "Completed executions unused after earlier finalization"
    ),
}
for limit, result in parallel_results_by_max_in_flight.items():
    print(f"\nParallel details (max in-flight samples: {max_in_flight_label(limit)})")
    for result_key, label in scheduler_detail_labels.items():
        print(f"{label}: {result[result_key]}")

Heavy-router training samples: 287
ResNet-50 route labels: 17
ResNet-152 route labels: 270
Excluded samples (neither heavy model meets the confidence threshold): 9999
Samples where both early models are below the threshold: 10286
Samples where both early models are below the threshold: 5099
Samples with a heavy-route target: 153
Excluded samples without an eligible heavy model: 4946
Heavy-route label accuracy: 0.8300653594771242
Target route counts: {'resnet50': 11, 'resnet152': 142}
Predicted route counts: {'resnet50': 33, 'resnet152': 120}
Confusion matrix (rows=target, columns=predicted; ResNet-50 then ResNet-152):
[[  9   2]
 [ 24 118]]
Running parallel scheduler (max in-flight samples: unlimited)
Running parallel scheduler (max in-flight samples: 2)
Running parallel scheduler (max in-flight samples: 3)
Running parallel scheduler (max in-flight samples: 4)
Running parallel scheduler (max in-flight samples: 8)
Running parallel scheduler (max in-flight samples: 16)
Running parallel s

## Real-Time Implementation


In [ ]:
import json
import tarfile
from pathlib import Path

from PIL import Image
import torch
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, IterableDataset
from torchvision import transforms

from real_time_mp_workers import model_worker

MAX_SAMPLES = 10000
BATCH_SIZE = 1
CPU_WORKERS = 1
MPS_WORKERS = 2
SAVE_RESULTS = True
MAX_IN_FLIGHT_PER_HEAVY_MODEL = 1

IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"
TEST_VARIANT = "threshold-0.7"
TEST_IMAGE_ARCHIVE = IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz"
RESULTS_PATH = RUNS_DIR / "real_cpu_mps_worker_results.json"
PREDICTIONS_PATH = RUNS_DIR / "real_cpu_mps_worker_predictions.npz"

CPU_MODELS = (RESNET18,)
MPS_MODELS = (RESNET34, RESNET50)
REAL_MODELS = CPU_MODELS + MPS_MODELS
REAL_ROUTE_LABEL_BY_MODEL = {RESNET34: 0, RESNET50: 1}
REAL_MODEL_BY_ROUTE_LABEL = {label: model for model, label in REAL_ROUTE_LABEL_BY_MODEL.items()}
CONFIDENCE_THRESHOLD = DYNAMIC_CONFIDENCE_THRESHOLD

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this CPU/MPS worker test")

MODEL_DEVICES = {
    RESNET18: "cpu",
    RESNET34: "mps",
    RESNET50: "mps",
}


def build_resnet18_heavy_router_dataset(caches, confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD, require_correct=False):
    validate_model_caches(caches)
    resnet18_features = extract_probability_features(caches[RESNET18]["probabilities"])
    true_labels = caches[RESNET18]["labels"]
    resnet18_is_uncertain = resnet18_features[:, 0] < confidence_threshold
    resnet34_is_confident = max_confidence(caches[RESNET34]) >= confidence_threshold
    resnet50_is_confident = max_confidence(caches[RESNET50]) >= confidence_threshold

    if require_correct:
        resnet34_is_eligible = resnet34_is_confident & (caches[RESNET34]["predictions"] == true_labels)
        resnet50_is_eligible = resnet50_is_confident & (caches[RESNET50]["predictions"] == true_labels)
    else:
        resnet34_is_eligible = resnet34_is_confident
        resnet50_is_eligible = resnet50_is_confident

    included = resnet18_is_uncertain & (resnet34_is_eligible | resnet50_is_eligible)
    route_labels = np.where(
        resnet34_is_eligible[included],
        REAL_ROUTE_LABEL_BY_MODEL[RESNET34],
        REAL_ROUTE_LABEL_BY_MODEL[RESNET50],
    ).astype(np.int64)
    return resnet18_features[included], route_labels


def train_resnet18_heavy_router(training_cache_sets, confidence_threshold=DYNAMIC_CONFIDENCE_THRESHOLD, require_correct=False):
    feature_parts = []
    label_parts = []
    for caches in training_cache_sets:
        features, labels = build_resnet18_heavy_router_dataset(caches, confidence_threshold, require_correct)
        if len(labels):
            feature_parts.append(features)
            label_parts.append(labels)
    if not label_parts:
        raise ValueError("No eligible samples were found for RN18-only heavy-router training")

    router = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
    )
    router.fit(np.concatenate(feature_parts), np.concatenate(label_parts))
    return router


real_heavy_router = train_resnet18_heavy_router(training_cache_sets)

print("CPU models:", ", ".join(CPU_MODELS))
print("MPS models:", ", ".join(MPS_MODELS))
print("Worker limits:", f"cpu={CPU_WORKERS}", f"mps={MPS_WORKERS}")


In [ ]:
preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


class ImageNetV2TarDataset(IterableDataset):
    def __init__(self, archive, transform, max_samples):
        self.archive = archive
        self.transform = transform
        self.max_samples = int(max_samples)

    def __iter__(self):
        emitted = 0
        with tarfile.open(self.archive, "r:*") as tar:
            for member in tar:
                if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                    continue

                image_file = tar.extractfile(member)
                if image_file is None:
                    continue
                image = Image.open(image_file).convert("RGB")
                image_file.close()

                yield self.transform(image), label_from_key(member.name)
                emitted += 1
                if emitted >= self.max_samples:
                    break

    def __len__(self):
        return self.max_samples


real_test_dataset = ImageNetV2TarDataset(TEST_IMAGE_ARCHIVE, preprocess, MAX_SAMPLES)
real_sample_count = len(real_test_dataset)
real_test_loader = DataLoader(
    real_test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=False,
)

print("Loaded samples:", real_sample_count)
print("Dataset:", TEST_IMAGE_ARCHIVE)
print("Worker model devices:", ", ".join(f"{name}={MODEL_DEVICES[name]}" for name in REAL_MODELS))


In [140]:
real_system_run_complete = False
real_system_results = None
real_final_predictions = None
real_chosen_models = None
router_call_count = 0
router_time_ms_total = 0.0


def route_heavy_model_from_resnet18(probabilities):
    features = extract_probability_features(probabilities[None, :])
    route_label = int(real_heavy_router.predict(features)[0])
    return REAL_MODEL_BY_ROUTE_LABEL[route_label]


def finish_sample(sample_index, model_name, prediction):
    final_predictions[sample_index] = prediction
    chosen_models[sample_index] = model_name
    latencies_ms[sample_index] = (
        time.perf_counter() - sample_states[sample_index]["start_time"]
    ) * 1000.0
    sample_states.pop(sample_index)


class HeavyNode:
    def __init__(self, sample_index, images, route_model):
        self.sample_index = sample_index
        self.images = images
        self.route_model = route_model
        self.prev = None
        self.next = None


class HeavyDoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        # partition points to the last RN34 node.
        # Nodes after partition are RN50.
        # If partition is None, the list has no RN34 jobs.
        self.partition = None
        self.size = 0

    def _append_empty(self, node):
        self.head = node
        self.tail = node
        self.size = 1
        if node.route_model == RESNET34:
            self.partition = node
        elif node.route_model == RESNET50:
            self.partition = None
        else:
            raise ValueError(f"Unsupported heavy route model: {node.route_model}")

    def _insert_before(self, anchor, node):
        node.prev = anchor.prev
        node.next = anchor

        if anchor.prev is None:
            self.head = node
        else:
            anchor.prev.next = node

        anchor.prev = node
        self.size += 1

    def _insert_after(self, anchor, node):
        node.prev = anchor
        node.next = anchor.next

        if anchor.next is None:
            self.tail = node
        else:
            anchor.next.prev = node

        anchor.next = node
        self.size += 1

    def _append_tail(self, node):
        if self.tail is None:
            self._append_empty(node)
            return

        self._insert_after(self.tail, node)

    def _remove_node(self, node):
        if node.prev is None:
            self.head = node.next
        else:
            node.prev.next = node.next

        if node.next is None:
            self.tail = node.prev
        else:
            node.next.prev = node.prev

        if node is self.partition:
            self.partition = node.prev

        node.prev = None
        node.next = None
        self.size -= 1

        if self.size == 0:
            self.head = None
            self.tail = None
            self.partition = None

        return node

    def insert_middle(self, sample_index, images, route_model):
        if route_model not in (RESNET34, RESNET50):
            raise ValueError(f"Unsupported heavy route model: {route_model}")

        node = HeavyNode(sample_index, images, route_model)

        if self.size == 0:
            self._append_empty(node)
            return node

        if route_model == RESNET34:
            # Insert RN34 at the boundary between RN34 and RN50.
            if self.partition is None:
                # There are no RN34 jobs yet, so put this at the head.
                self._insert_before(self.head, node)
            else:
                # Put this after the last RN34 job.
                self._insert_after(self.partition, node)

            self.partition = node

        elif route_model == RESNET50:
            # Insert RN50 immediately after the partition pointer.
            # This keeps the RN50 region right after RN34.
            # It is O(1), but RN50 becomes newest-first.
            if self.partition is None:
                # No RN34 jobs exist, so the whole list is RN50.
                # Put the new RN50 at the head.
                self._insert_before(self.head, node)
            else:
                # Put the new RN50 right after the last RN34.
                self._insert_after(self.partition, node)

        return node

    def pop_head_for_resnet34(self):
        # Since the list is partitioned, RN34 can only be at the head side.
        if self.head is None or self.head.route_model != RESNET34:
            return None

        node = self.head

        if node is self.partition:
            # This was the last RN34 job.
            self.partition = None

        return self._remove_node(node)


    def pop_tail_for_resnet50_or_steal_resnet34(self):
        # First priority: RN50 should process real RN50 jobs.
        if self.tail is not None and self.tail.route_model == RESNET50:
            return self._remove_node(self.tail), False

        # If there are no RN50 jobs waiting, RN50 can help with RN34 jobs.
        node = self.pop_head_for_resnet34()
        if node is None:
            return None, False

        return node, True


    # def pop_tail_for_resnet50(self):
    # # Since the list is partitioned, RN50 can only be on the tail side.
    #     if self.tail is None or self.tail.route_model != RESNET50:
    #         return None

    #     node = self.tail
    #     return self._remove_node(node)


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)
    ctx = mp.get_context("spawn")
    job_queues = {model_name: ctx.SimpleQueue() for model_name in REAL_MODELS}
    result_queue = ctx.SimpleQueue()
    processes = [
        ctx.Process(
            target=model_worker,
            args=(model_name, MODEL_DEVICES[model_name], job_queues[model_name], result_queue),
        )
        for model_name in REAL_MODELS
    ]

    for process in processes:
        process.start()

    try:
        total_samples = len(real_test_dataset)
        labels = np.full(total_samples, -1, dtype=np.int64)
        final_predictions = np.full(total_samples, -1, dtype=np.int64)
        chosen_models = np.full(total_samples, "", dtype="<U16")
        latencies_ms = np.full(total_samples, np.nan, dtype=np.float64)
        execution_count_by_model = Counter()
        execution_time_ms_by_model = Counter()
        heavy_route_count_by_model = Counter()
        in_flight_by_model = Counter()
        sample_states = {}
        heavy_waiting_list = HeavyDoublyLinkedList()
        heavy_queue_max_size = 0
        next_sample_index = 0
        completed_sample_count = 0
        loader_iterator = iter(real_test_loader)
        run_start = time.perf_counter()

        def current_heavy_backlog_size():
            return heavy_waiting_list.size + sum(
                in_flight_by_model[model_name] for model_name in MPS_MODELS
            )

        stolen_job_count_by_model = Counter()

        def dispatch_heavy_jobs(heavy_queue_max_size):
            for model_name in MPS_MODELS:
                while in_flight_by_model[model_name] < MAX_IN_FLIGHT_PER_HEAVY_MODEL:
                    if model_name == RESNET34:
                        node = heavy_waiting_list.pop_head_for_resnet34()
                        stole_rn34_job = False

                    elif model_name == RESNET50:
                        node, stole_rn34_job = heavy_waiting_list.pop_tail_for_resnet50_or_steal_resnet34()

                    else:
                        raise ValueError(f"Unsupported heavy model: {model_name}")

                    if node is None:
                        break

                    job_queues[model_name].put((node.sample_index, node.images))
                    in_flight_by_model[model_name] += 1
                    execution_count_by_model[model_name] += 1

                    if stole_rn34_job:
                        stolen_job_count_by_model[RESNET50] += 1

                    heavy_queue_max_size = max(
                        heavy_queue_max_size,
                        current_heavy_backlog_size(),
                    )

            return heavy_queue_max_size

        while completed_sample_count < total_samples:
            while in_flight_by_model[RESNET18] < CPU_WORKERS and next_sample_index < total_samples:
                images, batch_labels = next(loader_iterator)
                sample_index = next_sample_index
                next_sample_index += 1
                labels[sample_index] = int(batch_labels.item())
                sample_states[sample_index] = {
                    "images": images,
                    "start_time": time.perf_counter(),
                }
                job_queues[RESNET18].put((sample_index, images))
                in_flight_by_model[RESNET18] += 1
                execution_count_by_model[RESNET18] += 1

            message = result_queue.get()
            if message[0] == "error":
                _, model_name, error = message
                raise RuntimeError(f"{model_name} worker failed: {error}")

            sample_index, model_name, probabilities, prediction, confidence, elapsed_ms = message
            in_flight_by_model[model_name] -= 1
            execution_time_ms_by_model[model_name] += elapsed_ms

            if model_name == RESNET18 and confidence < CONFIDENCE_THRESHOLD:
                router_start = time.perf_counter()
                route_model = route_heavy_model_from_resnet18(probabilities)
                router_time_ms_total += (time.perf_counter() - router_start) * 1000.0
                router_call_count += 1  
                heavy_route_count_by_model[route_model] += 1
                heavy_waiting_list.insert_middle(
                    sample_index,
                    sample_states[sample_index]["images"],
                    route_model,
                )
                sample_states[sample_index]["images"] = None
                heavy_queue_max_size = max(
                    heavy_queue_max_size,
                    current_heavy_backlog_size(),
                )
            else:
                finish_sample(sample_index, model_name, prediction)
                completed_sample_count += 1

            heavy_queue_max_size = dispatch_heavy_jobs(heavy_queue_max_size)

        total_wall_time_seconds = time.perf_counter() - run_start
        correct_predictions = int(np.count_nonzero(final_predictions == labels))
        final_prediction_count_by_model = {
            model_name: int(np.count_nonzero(chosen_models == model_name))
            for model_name in REAL_MODELS
        }
        total_execution_time_ms_by_model = {
            model_name: float(execution_time_ms_by_model[model_name])
            for model_name in REAL_MODELS
        }
        idle_time_ms_by_model = {
            model_name: max(0.0, total_wall_time_seconds * 1000.0 - total_execution_time_ms_by_model[model_name])
            for model_name in REAL_MODELS
        }
        mean_execution_time_ms_by_model = {
            model_name: float(execution_time_ms_by_model[model_name] / execution_count_by_model[model_name])
            if execution_count_by_model[model_name]
            else 0.0
            for model_name in REAL_MODELS
        }

        real_final_predictions = final_predictions
        real_chosen_models = chosen_models
        real_system_results = {
            "total_samples": total_samples,
            "accuracy": correct_predictions / total_samples,
            "correct_predictions": correct_predictions,
            "total_wall_time_seconds": total_wall_time_seconds,
            "throughput_fps": total_samples / total_wall_time_seconds,
            "mean_latency_ms": float(latencies_ms.mean()),
            "p50_latency_ms": float(np.percentile(latencies_ms, 50)),
            "p95_latency_ms": float(np.percentile(latencies_ms, 95)),
            "device_by_model": {
                model_name: str(MODEL_DEVICES[model_name])
                for model_name in REAL_MODELS
            },
            "worker_limits": {"cpu": CPU_WORKERS, "mps": MPS_WORKERS},
            "final_prediction_count_by_model": final_prediction_count_by_model,
            "execution_count_by_model": {
                model_name: int(execution_count_by_model[model_name])
                for model_name in REAL_MODELS
            },
            "total_execution_time_ms_by_model": total_execution_time_ms_by_model,
            "idle_time_ms_by_model": idle_time_ms_by_model,
            "mean_execution_time_ms_by_model": mean_execution_time_ms_by_model,
            "heavy_route_count_by_model": {
                model_name: int(heavy_route_count_by_model[model_name])
                for model_name in MPS_MODELS
            },
            "heavy_queue_max_size": int(heavy_queue_max_size),
            "stolen_job_count_by_model": {
                model_name: int(stolen_job_count_by_model[model_name])
                for model_name in REAL_MODELS
            },
            "router_call_count": int(router_call_count),
            "total_router_time_ms": float(router_time_ms_total),
            "mean_router_time_ms": (
            float(router_time_ms_total / router_call_count)
            if router_call_count
            else 0.0
        ),
        }
        real_system_run_complete = True
    finally:
        for queue in job_queues.values():
            queue.put(None)
        for process in processes:
            process.join()


In [141]:
if not real_system_run_complete:
    raise RuntimeError("The real-system run did not complete; saved files were not changed")

print("Real-Time CPU/MPS Multiprocessing Test")
print("Worker limits:", real_system_results["worker_limits"])
print("Device by model:", real_system_results["device_by_model"])
print("Total samples:", real_system_results["total_samples"])
print("Accuracy:", round(real_system_results["accuracy"], 4))
print("Correct predictions:", real_system_results["correct_predictions"])
print("Total wall time (seconds):", round(real_system_results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(real_system_results["throughput_fps"], 3))
print("Mean latency (ms):", round(real_system_results["mean_latency_ms"], 3))
print("")
print("Random forest router:")
print(f"  Router calls: {real_system_results['router_call_count']}")
print(f"  Total router time (ms): {real_system_results['total_router_time_ms']:.3f}")
print(f"  Mean router time (ms): {real_system_results['mean_router_time_ms']:.6f}")
print("")

print(
    "# of jobs stolen by RN50 originally assigned to RN34:",
    real_system_results["stolen_job_count_by_model"][RESNET50],
)

print("")


print("Final prediction count by model:")
for model_name in REAL_MODELS:
    print(f"  {model_name}: {real_system_results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in REAL_MODELS:
    print(f"  {model_name}: {real_system_results['execution_count_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in REAL_MODELS:
    print(f"  {model_name}: {real_system_results['mean_execution_time_ms_by_model'][model_name]:.3f}")
print("Idle time by model (seconds):")
for model_name in REAL_MODELS:
    print(f"  {model_name}: {real_system_results['idle_time_ms_by_model'][model_name] / 1000.0:.3f}")
print("Heavy route count by MPS model:")
for model_name in MPS_MODELS:
    print(f"  {model_name}: {real_system_results['heavy_route_count_by_model'][model_name]}")
print("Heavy queue max size:", real_system_results["heavy_queue_max_size"])

if SAVE_RESULTS:
    RESULTS_PATH.write_text(json.dumps(real_system_results, indent=2) + " ", encoding="utf-8")
    np.savez_compressed(
        PREDICTIONS_PATH,
        predictions=real_final_predictions,
        chosen_models=real_chosen_models,
    )
    print("Saved:", RESULTS_PATH)
    print("Saved:", PREDICTIONS_PATH)
else:
    print("SAVE_RESULTS is False; no files were written.")


Real-Time CPU/MPS Multiprocessing Test
Worker limits: {'cpu': 1, 'mps': 2}
Device by model: {'resnet18': 'cpu', 'resnet34': 'mps', 'resnet50': 'mps'}
Total samples: 10000
Accuracy: 0.7337
Correct predictions: 7337
Total wall time (seconds): 216.593
Throughput (FPS): 46.17
Mean latency (ms): 33.438

Random forest router:
  Router calls: 6278
  Total router time (ms): 27055.997
  Mean router time (ms): 4.309652

# of jobs stolen by RN50 originally assigned to RN34: 416

Final prediction count by model:
  resnet18: 3722
  resnet34: 4001
  resnet50: 2277
Execution count by model:
  resnet18: 10000
  resnet34: 4001
  resnet50: 2277
Mean execution time by model (ms):
  resnet18: 14.579
  resnet34: 17.369
  resnet50: 29.840
Idle time by model (seconds):
  resnet18: 70.802
  resnet34: 147.098
  resnet50: 148.647
Heavy route count by MPS model:
  resnet34: 4417
  resnet50: 1861
Heavy queue max size: 8
Saved: real_cpu_mps_worker_results.json
Saved: real_cpu_mps_worker_predictions.npz
